# 03 — Model Training
**Dataset:** Fruits 360  
**Tujuan Notebook:** Melatih dan membandingkan beberapa model ML klasik untuk image classification.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle
import os

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = 'data/processed'
MODEL_DIR  = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)
print('Libraries loaded!')

## 2. Load Fitur dari Notebook 02

In [ ]:
X_train = np.load(f'{OUTPUT_DIR}/X_train_pca.npy')
X_test  = np.load(f'{OUTPUT_DIR}/X_test_pca.npy')
y_train = np.load(f'{OUTPUT_DIR}/y_train.npy')
y_test  = np.load(f'{OUTPUT_DIR}/y_test.npy')
label_names = pd.read_csv(f'{OUTPUT_DIR}/label_names.csv')['class_name'].tolist()

# Standarisasi fitur (penting untuk SVM dan KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'X_train shape: {X_train_scaled.shape}')
print(f'X_test  shape: {X_test_scaled.shape}')
print(f'Jumlah kelas : {len(np.unique(y_train))}')

## 3. Definisi Model

In [ ]:
models = {
    'SVM (RBF)': SVC(
        kernel='rbf', C=10, gamma='scale', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42, n_jobs=-1
    ),
    'KNN (k=5)': KNeighborsClassifier(
        n_neighbors=5, metric='euclidean', n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=1.0, random_state=42, n_jobs=-1
    ),
}

print('Model yang akan dilatih:')
for name in models:
    print(f'  - {name}')

## 4. Training & Evaluasi Semua Model

In [ ]:
results = []

for name, model in models.items():
    print(f'\nTraining: {name}...')
    start = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start

    # Prediksi
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test  = model.predict(X_test_scaled)

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc  = accuracy_score(y_test,  y_pred_test)

    print(f'  Train Accuracy : {train_acc:.4f}')
    print(f'  Test  Accuracy : {test_acc:.4f}')
    print(f'  Training Time  : {train_time:.2f}s')

    results.append({
        'Model': name,
        'Train Accuracy': round(train_acc, 4),
        'Test Accuracy':  round(test_acc, 4),
        'Training Time (s)': round(train_time, 2)
    })

    # Simpan model
    fname = name.replace(' ', '_').replace('(', '').replace(')', '').replace('=', '')
    with open(f'{MODEL_DIR}/{fname}.pkl', 'wb') as f:
        pickle.dump(model, f)

results_df = pd.DataFrame(results)
print('\n=== Perbandingan Semua Model ===')
print(results_df.to_string(index=False))

## 5. Visualisasi Perbandingan Model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
x = np.arange(len(results_df))
width = 0.35
axes[0].bar(x - width/2, results_df['Train Accuracy'], width, label='Train', color='steelblue')
axes[0].bar(x + width/2, results_df['Test Accuracy'],  width, label='Test',  color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['Model'], rotation=15, ha='right')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Train vs Test Accuracy per Model')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# Training time comparison
axes[1].bar(results_df['Model'], results_df['Training Time (s)'], color='mediumseagreen')
axes[1].set_ylabel('Waktu (detik)')
axes[1].set_title('Training Time per Model')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=100)
plt.show()

## 6. Pilih Model Terbaik

In [ ]:
best_row = results_df.loc[results_df['Test Accuracy'].idxmax()]
best_name = best_row['Model']
best_acc  = best_row['Test Accuracy']

print(f'Model terbaik : {best_name}')
print(f'Test Accuracy : {best_acc:.4f}')

# Simpan scaler
with open(f'{MODEL_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Simpan hasil perbandingan
results_df.to_csv(f'{OUTPUT_DIR}/model_comparison.csv', index=False)
print('Hasil perbandingan disimpan!')

## 7. Ringkasan
- 4 model ML klasik berhasil dilatih: SVM, Random Forest, KNN, Logistic Regression
- Semua model menggunakan fitur PCA (100 komponen) dari HOG + Color Histogram
- Model terbaik akan di-tuning lebih lanjut di notebook `04_hyperparameter_tuning.ipynb`